In [ ]:
!pip install transformers yfinance torch torchvision torchaudio scikit-learn pandas tqdm

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm


In [ ]:
# Example: Replace with your big dataset
df = pd.read_csv("your_news_dataset.csv")

# Expect at least: "text" and "label" columns
print(df.head())
print(df['label'].value_counts())


In [ ]:
# Load FinBERT
finbert = AutoModelForSequenceClassification.from_pretrained(
    "yiyanghkust/finbert-tone"
)
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")

finbert.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finbert.to(device)

sentiment_scores = []
batch_size = 16

with torch.no_grad():
    for i in tqdm(range(0, len(df), batch_size)):
        texts = df["text"].iloc[i:i+batch_size].tolist()
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = finbert(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        sentiment_scores.extend(probs)

# Add sentiment probabilities to dataframe
sentiment_df = pd.DataFrame(sentiment_scores, columns=["negative", "neutral", "positive"])
df = pd.concat([df.reset_index(drop=True), sentiment_df], axis=1)

print(df.head())


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Example: Use only FinBERT sentiment scores
X = df[["negative", "neutral", "positive"]]
y = df["label"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_dataset = SentimentDataset(X_train, y_train)
val_dataset = SentimentDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, num_layers=2, output_dim=3, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, seq_len=1, features)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMClassifier().to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 10
best_val_acc = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(y_batch.cpu().numpy())
    
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1:02d} | Train Loss {train_loss/len(train_loader):.4f} | Val Acc {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_lstm_model.pth")


In [ ]:
# If you have a test set:
# test_dataset = SentimentDataset(X_test, y_test)
# test_loader = DataLoader(test_dataset, batch_size=32)

# model.load_state_dict(torch.load("best_lstm_model.pth"))
# model.eval()

# preds, labels = [], []
# with torch.no_grad():
#     for X_batch, y_batch in test_loader:
#         X_batch, y_batch = X_batch.to(device), y_batch.to(device)
#         outputs = model(X_batch)
#         preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
#         labels.extend(y_batch.cpu().numpy())

# print("Test Accuracy:", accuracy_score(labels, preds))
# print(classification_report(labels, preds))
